In [1]:
import trino
import pandas as pd
from datetime import datetime

In [2]:
conn = trino.dbapi.connect(
    host="trino",
    port=8080,
    user="jupyter",
    catalog="iceberg"
)

In [4]:
cur = conn.cursor()

In [5]:
cur.execute("SELECT count(*) FROM serving_db.ma7").fetchall()

[[96]]

In [6]:
ctas_query = f"""
CREATE TABLE IF NOT EXISTS serving_db.sma7
WITH (
    format = 'PARQUET',
    location = 's3a://crypto-data-lake/serving_zone/sma7'
) AS
select 
    *,
    round((sum(close_price) over(order by group_id rows between 6 preceding and current row)) / 7, 2) as ma7
from serving_db.klines
offset 6
"""
cur.execute(ctas_query)

In [7]:
cur.execute("SELECT * FROM serving_db.sma7")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,group_id,group_date,open_time,open_price,high_price,low_price,close_price,volume,close_time,ma7
0,1948902,2025-08-01 01:30:00,1754011800063081,115190.37,115413.91,115000.00,115296.45,267.388,1754012699912234,115313.57
1,1948903,2025-08-01 01:45:00,1754012700223622,115296.46,115407.71,115060.11,115331.86,258.534,1754013599937883,115316.26
2,1948904,2025-08-01 02:00:00,1754013600005133,115328.67,115600.00,115221.07,115600.00,164.012,1754014499986628,115287.69
3,1948905,2025-08-01 02:15:00,1754014500063435,115600.00,115810.71,115511.60,115619.94,168.411,1754015399984518,115302.26
4,1948906,2025-08-01 02:30:00,1754015400022016,115619.95,116019.30,115572.53,115900.01,221.423,1754016299869454,115369.79
5,1948907,2025-08-01 02:45:00,1754016300091550,115900.01,116019.06,115843.42,115966.12,151.109,1754017199928964,115557.82
6,1948908,2025-08-01 03:00:00,1754017200094588,115966.11,115990.69,115794.14,115816.00,204.915,1754018099924949,115647.20
7,1948909,2025-08-01 03:15:00,1754018100126077,115816.00,116052.00,115816.00,116020.38,124.544,1754018999884991,115750.62
8,1948910,2025-08-01 03:30:00,1754019000385085,116020.39,116035.61,115828.61,115842.50,119.948,1754019899994161,115823.56
9,1948911,2025-08-01 03:45:00,1754019900527784,115842.50,115894.50,115613.21,115648.03,103.036,1754020799982068,115830.43
